# Proyecto de Estadística Multivariada
# Our World in Data CO2 and Greenhouse Gas Emissions dataset

## Librerías

In [35]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler



from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, silhouette_score

In [36]:
df_pca = pd.read_csv('df_pca_saved.csv')

## Regresión Lineal Múltiple (RLM)


Vamos a definir la variable a predecir "y" como 'co2' que son las emisiones de carbono y las variables predictoras serán: 'population', 'oil_co2', 'cement_co2' y 'gas_co2'. Se eligieron de esta manera por su relevancia teórica en comparación con las demás variables, además de su alta correlación con la variable "y" pero con correlación moderada entre ellas. Estas permiten modelar el impacto por sectores productivos y la demanda energética, por lo tanto son buenos estimadores.

### 1. Quitar los renglones vacíos de co2 (variable a predecir)

In [37]:
df_rlm = df_pca.copy()
df_rlm.dropna(subset=['co2'], inplace=True)

### 2. Definir variable objetivo y variables predictoras

In [38]:
# Variable objetivo (dependiente)
y = df_rlm['co2']

# Variables independientes (predictoras)
X = df_rlm[['population', 'oil_co2', 'cement_co2', 'gas_co2']]

X.shape, y.shape

((5100, 4), (5100,))

### 3. Dividir los datos en conjuntos de entrenamiento(80%) y prueba(20%)

In [39]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train.shape,X_test.shape

((4080, 4), (1020, 4))

### 4. Escalar variables

In [40]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### 5. Entrenar el modelo

In [41]:
model = LinearRegression()
model.fit(X_train_scaled, y_train)

LinearRegression()

### 6. Realizar predicciones

In [42]:
y_pred = model.predict(X_test_scaled)

### 7. Evaluar el modelo

In [43]:
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"R-squared (R2): {r2:.2f}")
print(f"Mean Absolute Error (MAE): {mae:.2f}")

Mean Squared Error (MSE): 7600.46
R-squared (R2): 0.99
Mean Absolute Error (MAE): 23.37


El R Squared de 0.97 es bastante alto, significa que el modelo explica el 97% de la variabilidad de las emisiones de CO2 basándose en las variables predictoras que utilizamos, es decir, refleja bien la estructura de los datos. Este resultado tiene sentido ya que 'oil_co2', 'cement_co2' y 'gas_co2' son componentes directos de las emisiones de CO2 total.

El Mean Absolute Error (MAE) nos dice (en las mismas unidades que CO2) cuánto se equivoca en promedio el modelo, en este caso un MAE de 27.34 quiere decir que en promedio el modelo se equivoca en unos 27.34 millones de toneladas de CO2. Recordemos que los rangos de emisiones van desde países que producen muy pocas toneladas de CO2 hasta países que producen miles de toneladas de CO2.

El Mean Squared Error (MSE) es de 4758.34. En este caso considerando las otras dos métricas, un R2 alto y MAE relativamente bajo, sugiere que el modelo es preciso.



### 8. Análisis de los coeficientes

In [44]:
feature_names = X.columns
coefficients = model.coef_

coef_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefficients})

coef_df = coef_df.sort_values(by='Coefficient', ascending=False)

print("Coeficientes del modelo de Regresión Lineal Múltiple:")
coef_df

Coeficientes del modelo de Regresión Lineal Múltiple:


,Feature,Coefficient
2,cement_co2,407.320812
0,population,94.130108
1,oil_co2,85.447662
3,gas_co2,48.102063


Los atributos cement_co2 (269.79), oil_co2 (255.53), gas_co2 (93.22) son los predictores más fuertes y positivos. Como era de esperar, un aumento en las emisiones de CO2 provenientes del cemento, el petróleo o el gas se traduce directamente en un aumento de las emisiones totales de CO2. El atributo population (30.12) tiene un coeficiente positivo pero menor que las fuentes de CO2, indica que, una desviación estándar en la población se asocia con un aumento de 30.13 unidades en las emisiones totales de CO2.

Este modelo al incluir directamente las principales fuentes de emisión de CO2 (oil_co2, cement_co2, gas_co2), junto con la población, logra un rendimiento sobresaliente.